In [1]:
import numpy as np
import cv2 as cv
import glob

rightImagesFolder = "stereoCalibrationRight"
leftImagesFolder = "stereoCalibrationLeft"

chessboardSize = (7,10) # internal corners of a 8 rows x 11 cols
chessboardTileSize = 20 # 20 mm tiles
frameSize = (640,480)

# Default criteria for termination of the camera calibration process recommended by the openCV documentation.
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)

objp = np.zeros((chessboardSize[0]*chessboardSize[1], 3), np.float32)
objp[:,:2] = np.mgrid[0:chessboardSize[0],0:chessboardSize[1]].T.reshape(-1,2)

objp = objp * chessboardTileSize


objPoints = [] # 3D point in real world space
imgPointsL = [] # 2D points in the image plane
imgPointsR = [] # 2D points in the image plane

In [2]:
imagesLeft = glob.glob(f"{leftImagesFolder}/*.jpg")
imagesRight = glob.glob(f"{rightImagesFolder}/*.jpg")

for imgLeft, imgRight in zip(imagesLeft, imagesRight):
    
    #Get images
    imgL = cv.imread(imgLeft)
    imgR = cv.imread(imgRight)

    #Convert them to Grayscale
    grayL = cv.cvtColor(imgL, cv.COLOR_BGR2GRAY)
    grayR = cv.cvtColor(imgR, cv.COLOR_BGR2GRAY)

    #Find the chessboard corners in the pictures
    retL, cornersL = cv.findChessboardCorners(grayL, chessboardSize, None)
    retR, cornersR = cv.findChessboardCorners(grayR, chessboardSize, None)

    #If both images have chessboards with corners that can be detectable
    if retL and retR == True:
        #Add the 3D world points
        objPoints.append(objp)

        #Refine 2D points in the image plane for subpixel precision
        cornersL = cv.cornerSubPix(grayL, cornersL, (11,11), (-1,-1), criteria)
        imgPointsL.append(cornersL)

        cornersR = cv.cornerSubPix(grayR, cornersR, (11,11), (-1,-1), criteria)
        imgPointsR.append(cornersR)

In [3]:
#Individual camera calibration
retL, cameraMatrixL, distL, rvecsL, tvecsL = cv.calibrateCamera(objPoints, imgPointsL, frameSize, None, None)
heightL, widthL, channelsL = imgL.shape
newCameraMatrixL, roiL = cv.getOptimalNewCameraMatrix(cameraMatrixL, distL, (widthL,heightL), 1, (widthL,heightL))

retR, cameraMatrixR, distR, rvecsR, tvecsR = cv.calibrateCamera(objPoints, imgPointsR, frameSize, None, None)
heightR, widthR, channelsR = imgR.shape
newCameraMatrixR, roiR = cv.getOptimalNewCameraMatrix(cameraMatrixR, distR, (widthR,heightR), 1, (widthR,heightR))

In [4]:
#Stereo Vision calibration
# This flag is important to specify that we already have the camera intrinsics estimated (camera matrixes and distortion coefficients),
# so for the Stereo Vision Calibration we just want to estimate the rotation, translation, essential and fundamental matrixes
flags = 0
flags |= cv.CALIB_FIX_INTRINSIC

# Default criteria for termination of the stereo vision calibration process recommended by the openCV documentation.
criteria_stereo = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 30, 0.001)

retStereo, newCameraMatrixL, distL, newCameraMatrixR, distR, rot, trans, essentialMatrix, fundamentalMatrix = cv.stereoCalibrate(
    objPoints,imgPointsL, imgPointsR, newCameraMatrixL, distL, newCameraMatrixR, distR, frameSize, flags=flags, criteria=criteria_stereo
     )

In [5]:
#Stereo Rectification

rectifyScale = 1
rectL, rectR, projMatrixL, projMatrixR, Q, roiL, roiR = cv.stereoRectify(
    newCameraMatrixL, distL, newCameraMatrixR, distR, grayL.shape[::1], rot, trans, rectifyScale, (0,0), alpha=1
    )

stereoMapL = cv.initUndistortRectifyMap(newCameraMatrixL, distL, rectL, projMatrixL, grayL.shape[::1], cv.CV_32FC1)
stereoMapR = cv.initUndistortRectifyMap(newCameraMatrixR, distR, rectR, projMatrixR, grayR.shape[::1], cv.CV_32FC1)

print("Parameters Saved!")
cv_file = cv.FileStorage('stereoMap.xml', cv.FileStorage_WRITE)

cv_file.write('q_matrix', Q)
cv_file.write('stereoMapL_x',stereoMapL[0])
cv_file.write('stereoMapL_y',stereoMapL[1])
cv_file.write('stereoMapR_x',stereoMapR[0])
cv_file.write('stereoMapR_y',stereoMapR[1])
cv_file.write('roiL', roiL)
cv_file.write('roiR', roiR)

cv_file.release()

print(roiL)
print(roiR)

Parameters Saved!
(259, 117, 176, 246)
(265, 121, 169, 226)
